**PART2a: The Anatomy of a Prompt**

In [1]:
!pip install -q langchain langchain-google-genai langchain-core python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.5 MB/s eta 0:00:00


In [2]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

# Using Low Temp for consistent comparison
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

Enter your Google API Key: ··········


In [3]:
task = "Write a leave letter to class teacher."

print("--- LAZY PROMPT ---")
print(llm.invoke(task).content)

--- LAZY PROMPT ---
Here are a few options for a leave letter to your class teacher, depending on who is writing it (you or your parent/guardian) and the reason for leave.

---

**Option 1: Written by a Parent/Guardian (Most Common)**

This is the most common and preferred way for a student to request leave, especially for younger students or for longer absences.

```
[Your Address/Parent's Address]
[City, Pincode]
[Date]

To,
The Class Teacher,
[Your Class/Section, e.g., Class 7-A]
[Your School Name]
[School Address/City]

Subject: Application for Leave of Absence - [Your Full Name]

Respected Ma'am/Sir,

This is to inform you that my child, [Your Full Name], a student of Class [Your Class/Section], will not be able to attend school from [Start Date] to [End Date] due to [Reason for leave - e.g., a severe cold and fever / a family wedding out of town / a doctor's appointment].

[He/She] is expected to return to school on [Date of Return].

I understand the importance of regular attend

In [5]:
structured_prompt = """
# Context
You are an student at a university called 'PES University'.

# Objective
Write a leave letter to ur class cordinatior.

# Constraints
1. Be extremely brief (under 50 words).
2. Do NOT say 'I will come after 2 days. Say 'why do u want a leave'.
3. Sign off with 'Yours, PES University'

# Output Format
Plain text, no subject line.
"""

print("--- STRUCTURED PROMPT ---")
print(llm.invoke(structured_prompt).content)

--- STRUCTURED PROMPT ---
Dear Coordinator,

I request leave on [Date(s)] due to personal reasons. I apologize for any inconvenience and will catch up on missed work.

Yours,
PES University


**Part 2b: Zero-Shot to Few-Shot**

In [6]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

In [7]:
prompt_zero = "Combine 'Angry' and 'Hungry' into a funny new word."
print(f"Zero-Shot: {llm.invoke(prompt_zero).content}")

Zero-Shot: The most common and widely accepted funny new word for being angry because you're hungry is **Hangry**.


In [8]:
prompt_few = """
Combine words into a funny new word. Give a sarcastic definition.

Input: Breakfast + Lunch
Output: Brunch (An excuse to drink alcohol before noon)

Input: Chill + Relax
Output: Chillax (What annoying people say when you are panic attacks)

Input: Angry + Hungry
Output:
"""
print(f"Few-Shot: {llm.invoke(prompt_few).content}")

Few-Shot: Input: Angry + Hungry
Output: Hangry (A medical condition that allows you to be an insufferable jerk to everyone around you, solely because you missed your afternoon snack.)


**Part 2c: Advanced Templates & Theory**

In [9]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [10]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# 1. Our Database of Examples
examples = [
    {"input": "The internet is down.", "output": "We are observing connectivity latency."},
    {"input": "This code implies a bug.", "output": "The logic suggests unintended behavior."},
    {"input": "I hate this feature.", "output": "This feature does not align with my preferences."},
]

# 2. Template for ONE example
example_fmt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

# 3. The Few-Shot Container
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_fmt,
    examples=examples
)

# 4. The Final Chain
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Corpo-Speak Translator. Rewrite the input to sound professional."),
    few_shot_prompt,      # Inject examples here
    ("human", "{text}")
])

chain = final_prompt | llm

print(chain.invoke({"text": "This app sucks."}).content)

The application presents areas for improvement.
